# 第 01 章：环境配置与智能体底层逻辑 (实战版)

根据讲义，本章实战目标：
- **统一模型接口**：使用 `init_chat_model` 消除厂商差异。
- **标准化输入**：掌握 `(role, content)` 元组列表的工业级组织方式。
- **流式处理**：理解 `astream` 的心智模型，掌握 `AIMessageChunk` 的聚合方式。

## 1. 统一模型声明与能力探针
配置环境变量，并使用通用接口实例化 LLM。

In [14]:
import requests
try:
    r = requests.get("https://api.deepseek.com", timeout=5)
    print(f"DeepSeek API 连通性测试成功：{r.status_code}")
except Exception as e:
    print(f"DeepSeek API 连通性测试失败: {e}")
    print("提示：如果连接超时或 SSL 错误，请检查网络代理或 VPN 配置。")

DeepSeek API 连通性测试成功：401


In [17]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 统一实例化：2026 年推荐做法
# 如果 api.deepseek.com 无法访问，可以尝试修改 base_url 或使用代理
llm = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    base_url="https://api.deepseek.com",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    streaming=True
)

print(f"模型加载成功：{llm.__class__.__name__}")

模型加载成功：ChatDeepSeek


## 2. 组织输入的“三层结构”
不再手动写散乱的消息，尝试使用一个构建函数来规范化输入。

In [18]:
def prepare_inputs(user_query: str, chat_history: list = None):
    """
    工业级输入组织模版
    """
    history = chat_history or []
    
    # 三层结构：System (人设) -> History (上下文) -> User (当前问题)
    messages = [
        ("system", "你是一个实事求是的助手。如果不知道时间，请使用工具。"),
        *history,
        ("user", user_query)
    ]
    
    return {"messages": messages}

example_input = prepare_inputs("你好，你是谁？")
print("组织后的输入结构：")
import pprint
pprint.pprint(example_input)

组织后的输入结构：
{'messages': [('system', '你是一个实事求是的助手。如果不知道时间，请使用工具。'), ('user', '你好，你是谁？')]}


## 3. 驱动模式：流式处理 (Standard Boilerplate)
我们将使用 `astream` (异步流) 处理输出。请注意观察 `full_response` 是如何通过 `+` 运算符自动聚合的。

In [19]:
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def get_system_time(query: str) -> str:
    """返回当前系统的具体时间。"""
    import datetime
    return f"北京时间：{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

agent = create_agent(
    model=llm,
    tools=[get_system_time],
)

async def run_streaming_demo(query):
    input_dict = prepare_inputs(query)
    full_response = None
    
    print(f"\n--- 提问: {query} ---\n")
    
    # 核心心智模型：流是一个广播频道
    async for part in agent.astream(input_dict, stream_mode="messages", version="v2"):
        if part["type"] == "messages":
            chunk, metadata = part["data"]
            
            # 1. 过滤：只展示发送自 'model' 节点的文本
            if metadata.get("langgraph_node") == "model" and chunk.content:
                print(chunk.content, end="", flush=True)
            
            # 2. 聚合：使用 + 运算符自动合并 Chunks
            if metadata.get("langgraph_node") == "model":
                full_response = chunk if full_response is None else full_response + chunk
                
    return full_response

# 在 Jupyter 中运行异步函数
final_msg = await run_streaming_demo("告诉我北京几点了？")
print(f"\n\n[聚合完毕] 消费 Token 总计: {final_msg.usage_metadata['total_tokens']}")


--- 提问: 告诉我北京几点了？ ---

我来帮您查询北京的当前时间。现在是北京时间：2026年4月12日 23:16:19（晚上11点16分19秒）。

[聚合完毕] 消费 Token 总计: 800


## 4. 总结 (Review)
1. **Input**: 使用 Tuple 列表组织，简单快捷。
2. **Metadata**: 使用 `langgraph_node` 过滤信号，避免“中间人”干扰。
3. **Aggregation**: 使用 `+` 合并 `AIMessageChunk`，优雅地拿到最终成品。